
# Template: Previsioni tradizionali e LSTM per serie temporali

> **Istruzioni**
> - Sostituisci i percorsi ai file e i nomi delle colonne nella sezione **Parametri**.
> - Esegui le celle nell'ordine: Import → Parametri → Caricamento dati → Sezione che ti interessa (Classico o LSTM).
> - Il template include sia regressione sia classificazione a seconda della metrica scelta.


In [ ]:

# ==== Import ====
import numpy as np
import pandas as pd
from pathlib import Path

# Visualizzazione
import matplotlib.pyplot as plt

# Modeli classici
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score, f1_score, classification_report
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# LSTM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(tf.__version__)


## Parametri

In [ ]:

# ==== Parametri da personalizzare ====
# Percorso al dataset
DATA_PATH = Path('/mnt/data/tuo_dataset.csv')  # <-- Cambia con il tuo file

# Nome della colonna bersaglio (target)
TARGET_COL = 'y'  # <-- Cambia con la tua variabile dipendente

# Colonne da usare come feature (se None, usa tutte tranne il target)
FEATURE_COLS = None  # per selezionare automaticamente tutte tranne TARGET_COL

# Se è una SERIE TEMPORALE: indicare la colonna data/time (opzionale ma consigliato)
DATETIME_COL = None  # es. 'timestamp'  (deve essere parseable come datetime)

# Tipo problema: 'reg' per regressione continua, 'clf' per classificazione
PROBLEM_TYPE = 'reg'

# Per LSTM (finestra temporale)
WINDOW_SIZE = 24      # numero di step passati da usare
HORIZON = 1           # quanti step in avanti prevedere
TEST_SIZE = 0.2       # proporzione del dataset per test
VAL_SIZE = 0.1        # proporzione all'interno del train per validazione
RANDOM_STATE = 42
BATCH_SIZE = 64
EPOCHS = 30


## Caricamento dati

In [ ]:

# ==== Caricamento ====
assert DATA_PATH.exists(), f"File non trovato: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)

if DATETIME_COL is not None and DATETIME_COL in df.columns:
    df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors='coerce')
    df = df.sort_values(DATETIME_COL).reset_index(drop=True)

# Selezione feature
if FEATURE_COLS is None:
    FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

print("Dimensioni dataset:", df.shape)
print("Target:", TARGET_COL)
print("Feature:", FEATURE_COLS[:10], "... (tot:", len(FEATURE_COLS), ")")
df.head()


## Parte A — Modelli classici (scikit-learn)

In [ ]:

# ==== Split train/test ====
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

# Se è serie temporale e vuoi evitare leakage, usa split temporale; altrimenti train_test_split
if DATETIME_COL is not None:
    # Split temporale semplice
    n = len(df)
    n_test = int(TEST_SIZE * n)
    X_train, y_train = X[:-n_test], y[:-n_test]
    X_test, y_test = X[-n_test:], y[-n_test:]
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True)

# Scaling (facoltativo ma consigliato per modelli lineari/logistici)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

if PROBLEM_TYPE == 'reg':
    # Regressione: LinearRegression + RandomForestRegressor
    models = {
        'LinearRegression': LinearRegression(),
        'RandomForestRegressor': RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=300)
    }
else:
    # Classificazione: LogisticRegression + RandomForestClassifier
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'RandomForestClassifier': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300)
    }

results = {}
for name, model in models.items():
    if 'Forest' in name:
        # gli alberi non richiedono scaling, ma usiamo X_train (non scalato) per coerenza
        model.fit(X_train, y_train) if PROBLEM_TYPE=='reg' else model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        # modelli lineari/logistici beneficiano dello scaling
        model.fit(X_train_s, y_train)
        y_pred = model.predict(X_test_s)

    if PROBLEM_TYPE == 'reg':
        rmse = mean_squared_error(y_test, y_pred, squared=False)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    else:
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        results[name] = {'Accuracy': acc, 'F1_weighted': f1}

pd.DataFrame(results).T


In [ ]:

# ==== Grafico (solo per regressione) ====
if PROBLEM_TYPE == 'reg':
    # usa il miglior modello in base a RMSE
    res_df = pd.DataFrame(results).T
    best_name = res_df['RMSE'].idxmin()
    best_model = LinearRegression() if best_name=='LinearRegression' else RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)
    if best_name=='LinearRegression':
        best_model.fit(scaler.fit_transform(X_train), y_train)
        y_hat = best_model.predict(scaler.transform(X_test))
    else:
        best_model.fit(X_train, y_train)
        y_hat = best_model.predict(X_test)

    plt.figure()
    plt.plot(y_test, label='Reale')
    plt.plot(y_hat, label='Predetto')
    plt.title(f'Confronto Reale vs Predetto — {best_name}')
    plt.legend()
    plt.show()
else:
    # Report di classificazione per il modello scelto (RandomForest)
    clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(classification_report(y_test, y_pred))


## Parte B — LSTM per serie temporali

In [ ]:

# ==== Preparazione dati per LSTM (finestratura) ====
assert DATETIME_COL is not None, "Per LSTM di serie temporali, specifica DATETIME_COL nelle impostazioni."

# Selezioniamo solo feature numeriche per semplicità
num_df = df.select_dtypes(include=[np.number]).copy()
assert TARGET_COL in num_df.columns, "Il target deve essere numerico o convertibile."
num_features = [c for c in num_df.columns if c != TARGET_COL]

# Scaling su TUTTE le feature + target (per inverse_transform sul target poi)
scaler_all = StandardScaler()
scaled = scaler_all.fit_transform(num_df[num_features + [TARGET_COL]])
scaled_df = pd.DataFrame(scaled, columns=num_features + [TARGET_COL])

def make_windows(data_array, target_index, window_size=24, horizon=1):
    Xs, ys = [], []
    for i in range(len(data_array) - window_size - horizon + 1):
        window = data_array[i:i+window_size, :]
        target_val = data_array[i+window_size+horizon-1, target_index]
        Xs.append(window)
        ys.append(target_val)
    return np.array(Xs), np.array(ys)

data_array = scaled_df.values
target_idx = scaled_df.columns.get_loc(TARGET_COL)
X_seq, y_seq = make_windows(data_array, target_idx, WINDOW_SIZE, HORIZON)

n_total = len(X_seq)
n_test = int(TEST_SIZE * n_total)
n_val = int(VAL_SIZE * (n_total - n_test))

X_train, y_train = X_seq[:-(n_test + n_val)], y_seq[:-(n_test + n_val)]
X_val, y_val = X_seq[-(n_test + n_val):-n_test], y_seq[-(n_test + n_val):-n_test]
X_test, y_test = X_seq[-n_test:], y_seq[-n_test:]

X_train.shape, X_val.shape, X_test.shape


In [ ]:

# ==== Modello LSTM ====
def build_lstm(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
    return model

model = build_lstm(X_train.shape[1:])
model.summary()


In [ ]:

# ==== Training ====
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss')
]
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=callbacks
)


In [ ]:

# ==== Valutazione ====
pred_test = model.predict(X_test).squeeze()

# Inverse scaling per riportare il target alla scala originale
# Costruiamo un array di placeholder per utilizzare inverse_transform sul target
def inverse_target(scaled_target_vals, scaler, target_col_name, feature_names):
    # scaler è stato fit su [num_features + target]
    # Ricostruiamo array vuoto e inseriamo i valori del target nella colonna giusta
    idx = feature_names + [target_col_name]
    target_index = len(idx) - 1
    dummy = np.zeros((len(scaled_target_vals), len(idx)))
    dummy[:, target_index] = scaled_target_vals
    inv = scaler.inverse_transform(dummy)
    return inv[:, target_index]

y_test_inv = inverse_target(y_test, scaler_all, TARGET_COL, num_features)
pred_test_inv = inverse_target(pred_test, scaler_all, TARGET_COL, num_features)

rmse = mean_squared_error(y_test_inv, pred_test_inv, squared=False)
mae = mean_absolute_error(y_test_inv, pred_test_inv)
print(f"RMSE (test): {rmse:.4f} | MAE (test): {mae:.4f}")


In [ ]:

# ==== Grafico pred vs reale ====
plt.figure()
plt.plot(y_test_inv, label='Reale')
plt.plot(pred_test_inv, label='Predetto (LSTM)')
plt.title('Confronto Reale vs Predetto — LSTM')
plt.legend()
plt.show()



### Note & consigli
- **Evitare leakage**: per serie temporali, non mischiare futuro con passato nello split.
- **Feature engineering**: valuta lag, rolling mean/std, differenze, dummy per stagionalità/ora.
- **Scelta finestra (`WINDOW_SIZE`)**: prova più valori (es. 24, 48, 168) in base alla stagionalità.
- **Classificazione con LSTM**: se il target è categorico, converti in interi o one-hot e usa `loss='sparse_categorical_crossentropy'` o `categorical_crossentropy` con `Dense(n_classi, activation='softmax')`.
- **Multi-step forecast**: aumenta `HORIZON` o usa strategie ricorsive.
- **Salvataggio modello**: `model.save('/mnt/data/lstm_model.keras')`.
